# AVNIT: Model 2 — Vehicle Color Classifier Training
### Architecture: Deep Convolutional Neural Network — MobileNetV3-Small (PyTorch Transfer Learning)
### Dataset: Vehicle Color Recognition Dataset (8 Classes: White, Black, Silver/Grey, Red, Blue, Yellow, Green, Brown)

This notebook trains a deep CNN vehicle color classification model for the **AVNIT** identity verification system on Google Colab's free T4 GPU.
It automatically mounts your **Google Drive**, creates a backup folder, saves checkpoints per epoch, and downloads `color_classifier.pt`.

---

## 1. Verify GPU Hardware Acceleration
Ensure runtime accelerator is set to **T4 GPU** (`Runtime` > `Change runtime type` > `T4 GPU`).

In [ ]:
!nvidia-smi

## 2. Mount Google Drive for Automatic Epoch Checkpointing
All trained model checkpoints and training logs are automatically backed up to `My Drive/AVNIT_Models/`.

In [ ]:
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Destination folder in your Google Drive
DRIVE_SAVE_DIR = '/content/drive/MyDrive/AVNIT_Models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"[AVNIT] Google Drive mounted. Model weights will be saved to: {DRIVE_SAVE_DIR}")

## 3. Install Dependencies
Install PyTorch, Torchvision, Pillow, and Matplotlib.

In [ ]:
!pip install torch torchvision pillow matplotlib tqdm scikit-learn

## 4. Plug-and-Play Dataset Setup (Vehicle Color Recognition)
Auto-downloads the curated vehicle color dataset or uses custom uploaded `vehicle_colors.zip`.

In [ ]:
import urllib.request
import zipfile

DATA_DIR = '/content/vehicle_colors'
os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists('/content/vehicle_colors.zip'):
    print("Using custom uploaded vehicle_colors.zip...")
    with zipfile.ZipFile('/content/vehicle_colors.zip', 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
else:
    print("Downloading curated Vehicle Color Recognition dataset...")
    url = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/vehicle-color-sample.zip'
    try:
        urllib.request.urlretrieve(url, '/content/colors.zip')
        with zipfile.ZipFile('/content/colors.zip', 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print("Color dataset extracted successfully!")
    except Exception as e:
        print("Download notice:", e)
        classes = ['white', 'black', 'silver_grey', 'red', 'blue', 'yellow', 'green', 'brown']
        for split in ['train', 'val']:
            for c in classes:
                os.makedirs(os.path.join(DATA_DIR, split, c), exist_ok=True)

train_dir = os.path.join(DATA_DIR, 'train')
val_dir = os.path.join(DATA_DIR, 'val')
print("Classes found:", os.listdir(train_dir) if os.path.exists(train_dir) else 'Ready')

## 5. MobileNetV3-Small Deep Neural Network Configuration
MobileNetV3 uses Depthwise Separable Convolutions and Squeeze-and-Excitation attention blocks.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("Training device:", device)

TARGET_CLASSES = ['white', 'black', 'silver_grey', 'red', 'blue', 'yellow', 'green', 'brown']
num_classes = len(TARGET_CLASSES)

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Pre-trained deep backbone
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print("Neural Network initialized. Total trainable parameters:", sum(p.numel() for p in model.parameters()))

## 6. Training Loop with Drive Checkpointing
Trains for 20 epochs on T4 GPU and updates the best checkpoint.

In [ ]:
if os.path.exists(train_dir) and any(os.scandir(train_dir)):
    train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
    val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms) if os.path.exists(val_dir) and any(os.scandir(val_dir)) else train_dataset
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

    epochs = 20
    best_acc = 0.0
    best_state_dict = None

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            total += labels.size(0)
        
        scheduler.step()
        epoch_acc = (correct.double() / total).item()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/total:.4f} - Accuracy: {epoch_acc*100:.2f}%")
        
        if epoch_acc >= best_acc:
            best_acc = epoch_acc
            best_state_dict = model.state_dict()
            # Save intermediate epoch checkpoint to Google Drive
            torch.save({'state_dict': best_state_dict, 'classes': TARGET_CLASSES}, os.path.join(DRIVE_SAVE_DIR, 'color_classifier_best_epoch.pt'))
else:
    print("Train directory empty. Saving pretrained initialization checkpoint...")
    best_state_dict = model.state_dict()

## 7. Save to Google Drive & Trigger Browser Download
Saves `color_classifier.pt` to your Google Drive and triggers a local download.

In [ ]:
from google.colab import files

checkpoint = {
    'state_dict': best_state_dict if best_state_dict is not None else model.state_dict(),
    'classes': TARGET_CLASSES,
    'architecture': 'mobilenet_v3_small'
}

output_file = 'color_classifier.pt'
torch.save(checkpoint, output_file)

# 1. Save to Google Drive
drive_dest = os.path.join(DRIVE_SAVE_DIR, output_file)
torch.save(checkpoint, drive_dest)
print(f"[AVNIT] Successfully backed up to Google Drive at: {drive_dest}")

# 2. Trigger browser download to PC
print(f"[AVNIT] Triggering browser download for {output_file}...")
files.download(output_file)